# Component 2 — Phonation-Based Parkinson's Detection (baseline)

Dataset: MDVR-KCL, `ReadText` task, PD vs HC (37 recordings: 16 PD / 21 HC).

This notebook runs entirely on the local repo — no Google Drive / Colab dependency.
All feature extraction lives in `src/component2_phonation/features.py` and is used
**identically** for training and for predicting on new recordings (the original
Colab notebook used two different, incompatible formulas for jitter/shimmer/HNR at
training vs. prediction time, which made real predictions unreliable — see
`src/component2_phonation/README.md`).

Known limitation: MDVR-KCL has no sustained-vowel ("aaaaaa") task, so `ReadText`
is used as a stand-in for this phase-1 English baseline.

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import pandas as pd
import matplotlib.pyplot as plt

from src.component2_phonation.dataset import build_readtext_features, ensure_extracted
from src.component2_phonation.features import FEATURE_NAMES

RESULTS_DIR = os.path.join(REPO_ROOT, "results", "component2_phonation")
FEATURES_CSV = os.path.join(RESULTS_DIR, "mdvr_kcl_readtext_features.csv")

## Load features

Loads the cached CSV produced by `python -m src.component2_phonation.train` if it
exists (fast). Otherwise extracts features from the audio directly (slow — several
minutes, since jitter/shimmer/HNR go through Praat via `parselmouth`).

In [ ]:
if os.path.exists(FEATURES_CSV):
    df = pd.read_csv(FEATURES_CSV)
    print(f"Loaded cached features from {FEATURES_CSV}")
else:
    ensure_extracted()
    df = build_readtext_features()
    os.makedirs(RESULTS_DIR, exist_ok=True)
    df.to_csv(FEATURES_CSV, index=False)

print("Shape:", df.shape)
print(df["label_name"].value_counts())
df.head()

## Exploratory analysis

In [ ]:
summary = df.groupby("label_name")[FEATURE_NAMES].mean().T
display(summary)

In [ ]:
plt.figure(figsize=(7, 5))
plt.boxplot(
    [df[df["label_name"] == "HC"]["jitter"], df[df["label_name"] == "PD"]["jitter"]],
    tick_labels=["HC", "PD"],
)
plt.ylabel("Jitter")
plt.title("Jitter: PD vs HC")
plt.show()

In [ ]:
corr = df[FEATURE_NAMES].corr()
plt.figure(figsize=(8, 6))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(FEATURE_NAMES)), FEATURE_NAMES, rotation=90)
plt.yticks(range(len(FEATURE_NAMES)), FEATURE_NAMES)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## Train + evaluate

The full pipeline (participant-grouped 5-fold CV across RF/SVM/LogReg, final model
fit, plots, metrics.json) lives in `src/component2_phonation/train.py` so it can be
re-run identically from the command line. Re-running it here refreshes
`models/component2_phonation_rf.joblib` and everything under `results/component2_phonation/`.

In [ ]:
from src.component2_phonation import train as train_module
train_module.main()

## Predict on a single recording

Uses the exact same `extract_phonation_features` function used in training —
no train/predict feature mismatch.

In [ ]:
from src.component2_phonation.predict import predict

sample_path = os.path.join(REPO_ROOT, "data", "raw", "26-29_09_2017_KCL", "ReadText", "PD", "ID02_pd_2_0_0.wav")
result = predict(sample_path)
print(result["audio_path"])
print("Prediction:", result["prediction"])
print(f"P(HC) = {result['probability_hc']:.4f}   P(PD) = {result['probability_pd']:.4f}")